# MLAAD v10 scan, split, and MFCC cache

This notebook prepares the audio experiment evidence for the report. It mounts Google Drive, scans MLAAD-style synthetic audio plus genuine audio, creates a leakage-aware split, and caches MFCC features for the SVM, MLP, and LSTM notebooks.


## Goal

Build report-ready dataset evidence and reusable MFCC feature caches for binary audio deepfake detection: `0 = bona-fide/genuine`, `1 = synthetic/deepfake`.


In [ ]:
# Bootstrap shared MLAAD helpers. This makes the notebook self-contained in Colab.
from pathlib import Path
import sys

HELPER_SOURCE = "\"\"\"Shared helpers for MLAAD v10 Colab MFCC experiments.\n\nThe notebooks in this project are intentionally Colab-first. They call these\nhelpers to scan MLAAD-style folders, build leakage-aware splits, cache MFCC\nfeatures to Drive, and write report-ready tables and figures.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Optional, Sequence, Tuple\n\nimport numpy as np\nimport pandas as pd\n\n\nAUDIO_EXTENSIONS = (\".wav\", \".flac\", \".mp3\", \".ogg\", \".m4a\")\nCLASS_NAMES = {0: \"bona_fide\", 1: \"synthetic\"}\nOUTPUT_SUBDIRS = (\"figures\", \"cached\", \"mfcc\", \"metrics\", \"models\", \"tables\")\nMLAAD_META_COLUMNS = [\n    \"path\",\n    \"original_file\",\n    \"language\",\n    \"is_original_language\",\n    \"duration\",\n    \"training_data\",\n    \"model_name\",\n    \"architecture\",\n    \"transcript\",\n    \"reference_speaker\",\n]\n\n\ndef install_missing_packages(packages: Sequence[Tuple[str, str]]) -> None:\n    \"\"\"Install missing packages inside Colab without failing local static checks.\n\n    Args:\n        packages: Sequence of (import_name, pip_name).\n    \"\"\"\n    import importlib.util\n\n    missing = [pip_name for import_name, pip_name in packages if importlib.util.find_spec(import_name) is None]\n    if not missing:\n        print(\"Required packages are already available.\")\n        return\n    print(\"Installing missing packages:\", \", \".join(missing))\n    subprocess.check_call([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", *missing])\n\n\ndef mount_drive_if_colab(mount_point: str = \"/content/drive\") -> None:\n    \"\"\"Mount Google Drive when running inside Colab.\"\"\"\n    try:\n        from google.colab import drive  # type: ignore\n    except Exception:\n        print(\"google.colab is not available; assuming Drive is already accessible or running locally.\")\n        return\n    drive.mount(mount_point)\n\n\ndef ensure_output_dirs(output_root: str | Path) -> Dict[str, Path]:\n    \"\"\"Create and return the required output directories.\"\"\"\n    output_root = Path(output_root)\n    dirs = {\"root\": output_root}\n    output_root.mkdir(parents=True, exist_ok=True)\n    for name in OUTPUT_SUBDIRS:\n        path = output_root / name\n        path.mkdir(parents=True, exist_ok=True)\n        dirs[name] = path\n    return dirs\n\n\ndef write_json(path: str | Path, payload: dict) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding=\"utf-8\")\n\n\ndef read_json(path: str | Path) -> dict:\n    return json.loads(Path(path).read_text(encoding=\"utf-8\"))\n\n\ndef stable_config_hash(payload: dict) -> str:\n    raw = json.dumps(payload, sort_keys=True, default=str).encode(\"utf-8\")\n    return hashlib.sha256(raw).hexdigest()[:16]\n\n\ndef is_audio_file(path: str | Path) -> bool:\n    return Path(path).suffix.lower() in AUDIO_EXTENSIONS\n\n\ndef file_exists_bool(path: str | Path) -> bool:\n    try:\n        return Path(path).exists()\n    except OSError:\n        return False\n\n\ndef _iter_audio_files(root: Path) -> Iterable[Path]:\n    for path in root.rglob(\"*\"):\n        if path.is_file() and is_audio_file(path):\n            yield path\n\n\ndef _relative_str(path: Path, root: Path) -> str:\n    try:\n        return path.relative_to(root).as_posix()\n    except ValueError:\n        return path.as_posix()\n\n\ndef _safe_str(value, default: str = \"unknown\") -> str:\n    if value is None:\n        return default\n    try:\n        if pd.isna(value):\n            return default\n    except Exception:\n        pass\n    text = str(value).strip()\n    return text if text else default\n\n\ndef _read_meta_csv(meta_path: Path) -> pd.DataFrame:\n    for sep in (\"|\", \",\", \"\\t\"):\n        try:\n            df = pd.read_csv(meta_path, sep=sep)\n        except Exception:\n            continue\n        if len(df.columns) > 1:\n            df.columns = [str(c).strip() for c in df.columns]\n            df[\"meta_csv_path\"] = meta_path.as_posix()\n            return df\n    return pd.DataFrame()\n\n\ndef load_mlaad_metadata(dataset_root: str | Path) -> pd.DataFrame:\n    \"\"\"Read all MLAAD meta.csv files below a dataset root.\"\"\"\n    dataset_root = Path(dataset_root)\n    frames = []\n    for meta_path in dataset_root.rglob(\"meta.csv\"):\n        df = _read_meta_csv(meta_path)\n        if not df.empty:\n            frames.append(df)\n    if not frames:\n        return pd.DataFrame(columns=MLAAD_META_COLUMNS + [\"meta_csv_path\"])\n    return pd.concat(frames, ignore_index=True, sort=False)\n\n\ndef _resolve_meta_audio_path(row: pd.Series, dataset_root: Path, meta_path: Path) -> Optional[Path]:\n    raw_path = _safe_str(row.get(\"path\"), default=\"\")\n    if not raw_path:\n        return None\n    raw_path = raw_path.replace(\"\\\\\", \"/\")\n    candidates = [\n        dataset_root / raw_path,\n        meta_path.parent / raw_path,\n        meta_path.parent / Path(raw_path).name,\n    ]\n    if raw_path.startswith(\"fake/\") and (dataset_root / \"fake\").exists():\n        candidates.append(dataset_root / raw_path)\n    for candidate in candidates:\n        if candidate.exists() and candidate.is_file():\n            return candidate.resolve()\n    return None\n\n\ndef _infer_synthetic_parts(audio_path: Path, dataset_root: Path) -> Tuple[str, str]:\n    try:\n        parts = list(audio_path.relative_to(dataset_root).parts)\n    except ValueError:\n        parts = list(audio_path.parts)\n    if parts and parts[0].lower() == \"fake\":\n        parts = parts[1:]\n    language = parts[0] if len(parts) >= 3 else (parts[-3] if len(parts) >= 3 else \"unknown\")\n    tts_generator = parts[1] if len(parts) >= 3 else audio_path.parent.name\n    return _safe_str(language), _safe_str(tts_generator)\n\n\ndef _infer_genuine_language(audio_path: Path, dataset_root: Path) -> str:\n    try:\n        parts = list(audio_path.relative_to(dataset_root).parts)\n    except ValueError:\n        parts = list(audio_path.parts)\n    if len(parts) >= 2:\n        return _safe_str(parts[0])\n    return \"unknown\"\n\n\ndef scan_synthetic_audio(mlaad_synthetic_dir: str | Path) -> pd.DataFrame:\n    \"\"\"Scan MLAAD synthetic audio and merge folder-level metadata where present.\"\"\"\n    from tqdm.auto import tqdm\n\n    dataset_root = Path(mlaad_synthetic_dir)\n    if not dataset_root.exists():\n        raise FileNotFoundError(f\"MLAAD synthetic directory does not exist: {dataset_root}\")\n\n    metadata = load_mlaad_metadata(dataset_root)\n    records: List[dict] = []\n    seen_paths = set()\n\n    if not metadata.empty:\n        for _, row in tqdm(metadata.iterrows(), total=len(metadata), desc=\"Reading MLAAD metadata\"):\n            meta_path = Path(_safe_str(row.get(\"meta_csv_path\"), default=\"\"))\n            audio_path = _resolve_meta_audio_path(row, dataset_root, meta_path)\n            if audio_path is None or not is_audio_file(audio_path):\n                continue\n            language, tts_generator = _infer_synthetic_parts(audio_path, dataset_root)\n            language = _safe_str(row.get(\"language\"), default=language)\n            tts_generator = _safe_str(row.get(\"model_name\"), default=tts_generator)\n            records.append(\n                {\n                    \"path\": audio_path.as_posix(),\n                    \"relative_path\": _relative_str(audio_path, dataset_root),\n                    \"label\": 1,\n                    \"class_name\": CLASS_NAMES[1],\n                    \"language\": language,\n                    \"tts_generator\": tts_generator,\n                    \"architecture\": _safe_str(row.get(\"architecture\")),\n                    \"duration_metadata\": row.get(\"duration\", np.nan),\n                    \"original_file\": _safe_str(row.get(\"original_file\")),\n                    \"reference_speaker\": _safe_str(row.get(\"reference_speaker\")),\n                    \"source_dataset\": \"MLAAD_10pct\",\n                }\n            )\n            seen_paths.add(audio_path.as_posix())\n\n    for audio_path in tqdm(list(_iter_audio_files(dataset_root)), desc=\"Scanning synthetic audio files\"):\n        audio_path = audio_path.resolve()\n        if audio_path.as_posix() in seen_paths:\n            continue\n        language, tts_generator = _infer_synthetic_parts(audio_path, dataset_root)\n        records.append(\n            {\n                \"path\": audio_path.as_posix(),\n                \"relative_path\": _relative_str(audio_path, dataset_root),\n                \"label\": 1,\n                \"class_name\": CLASS_NAMES[1],\n                \"language\": language,\n                \"tts_generator\": tts_generator,\n                \"architecture\": \"unknown\",\n                \"duration_metadata\": np.nan,\n                \"original_file\": \"unknown\",\n                \"reference_speaker\": \"unknown\",\n                \"source_dataset\": \"MLAAD_10pct\",\n            }\n        )\n\n    return pd.DataFrame(records)\n\n\ndef scan_genuine_audio(genuine_audio_dir: str | Path) -> pd.DataFrame:\n    \"\"\"Scan bona-fide/genuine audio files from a Drive folder.\"\"\"\n    from tqdm.auto import tqdm\n\n    dataset_root = Path(genuine_audio_dir)\n    if not dataset_root.exists():\n        raise FileNotFoundError(f\"Genuine audio directory does not exist: {dataset_root}\")\n\n    records = []\n    for audio_path in tqdm(list(_iter_audio_files(dataset_root)), desc=\"Scanning genuine audio files\"):\n        audio_path = audio_path.resolve()\n        records.append(\n            {\n                \"path\": audio_path.as_posix(),\n                \"relative_path\": _relative_str(audio_path, dataset_root),\n                \"label\": 0,\n                \"class_name\": CLASS_NAMES[0],\n                \"language\": _infer_genuine_language(audio_path, dataset_root),\n                \"tts_generator\": \"bona_fide\",\n                \"architecture\": \"human\",\n                \"duration_metadata\": np.nan,\n                \"original_file\": _relative_str(audio_path, dataset_root),\n                \"reference_speaker\": \"unknown\",\n                \"source_dataset\": \"genuine_audio\",\n            }\n        )\n    return pd.DataFrame(records)\n\n\ndef _infer_tiny_label(row: dict) -> Optional[int]:\n    audio_info = row.get(\"audio\", {})\n    path_hint = \"\"\n    if isinstance(audio_info, dict):\n        path_hint = _safe_str(audio_info.get(\"path\"), default=\"\")\n    for key in (\"path\", \"file\", \"audio_path\"):\n        if key in row:\n            path_hint = f\"{path_hint} {_safe_str(row.get(key), default='')}\"\n    lower_path = path_hint.lower()\n    if any(token in lower_path for token in (\"original\", \"bona\", \"genuine\", \"real\")):\n        return 0\n    if any(token in lower_path for token in (\"fake\", \"spoof\", \"synthetic\")):\n        return 1\n\n    for key in (\"label\", \"labels\", \"class\", \"class_name\", \"target\"):\n        if key not in row:\n            continue\n        value = row[key]\n        text = str(value).lower()\n        if any(token in text for token in (\"original\", \"bona\", \"genuine\", \"real\")):\n            return 0\n        if any(token in text for token in (\"fake\", \"spoof\", \"synthetic\")):\n            return 1\n        if isinstance(value, (int, np.integer)) and value in (0, 1):\n            return int(value)\n    return None\n\n\ndef build_mlaad_tiny_smoke_manifest(\n    output_root: str | Path,\n    max_files_per_class: int,\n    random_state: int = 42,\n) -> pd.DataFrame:\n    \"\"\"Create a tiny local smoke-test manifest from the Hugging Face MLAAD-tiny stream.\"\"\"\n    from datasets import load_dataset\n    import soundfile as sf\n    from tqdm.auto import tqdm\n\n    output_root = Path(output_root)\n    smoke_root = output_root / \"cached\" / \"mlaad_tiny_smoke_audio\"\n    manifest_path = output_root / \"cached\" / f\"mlaad_tiny_smoke_manifest_{max_files_per_class}.csv\"\n    if manifest_path.exists():\n        manifest = pd.read_csv(manifest_path)\n        if set(manifest[\"label\"].unique()) == {0, 1}:\n            print(f\"Reusing MLAAD-tiny smoke manifest: {manifest_path}\")\n            return manifest\n\n    smoke_root.mkdir(parents=True, exist_ok=True)\n    dataset = load_dataset(\"mueller91/MLAAD-tiny\", split=\"train\", streaming=True)\n    counts = {0: 0, 1: 0}\n    records = []\n\n    for row in tqdm(dataset, desc=\"Streaming MLAAD-tiny smoke rows\"):\n        label = _infer_tiny_label(row)\n        if label not in (0, 1) or counts[label] >= max_files_per_class:\n            if all(count >= max_files_per_class for count in counts.values()):\n                break\n            continue\n\n        audio_info = row.get(\"audio\", {})\n        if not isinstance(audio_info, dict) or audio_info.get(\"array\") is None:\n            continue\n        array = np.asarray(audio_info[\"array\"], dtype=np.float32)\n        sample_rate = int(audio_info.get(\"sampling_rate\") or 22050)\n        class_name = CLASS_NAMES[label]\n        out_dir = smoke_root / class_name\n        out_dir.mkdir(parents=True, exist_ok=True)\n        out_path = out_dir / f\"{class_name}_{counts[label]:05d}.wav\"\n        sf.write(out_path, array, sample_rate)\n\n        path_hint = _safe_str(audio_info.get(\"path\"), default=out_path.name)\n        generator = \"bona_fide\" if label == 0 else Path(path_hint).parent.name\n        if generator in (\"\", \".\", \"unknown\"):\n            generator = \"MLAAD_tiny_spoof\" if label == 1 else \"bona_fide\"\n        language = \"en\"\n        if label == 1 and \"de\" in path_hint.lower().split(\"/\"):\n            language = \"de\"\n\n        records.append(\n            {\n                \"path\": out_path.as_posix(),\n                \"relative_path\": out_path.relative_to(smoke_root).as_posix(),\n                \"label\": label,\n                \"class_name\": class_name,\n                \"language\": language,\n                \"tts_generator\": generator,\n                \"architecture\": \"unknown\" if label == 1 else \"human\",\n                \"duration_metadata\": row.get(\"duration\", np.nan),\n                \"original_file\": path_hint if label == 1 else out_path.name,\n                \"reference_speaker\": \"unknown\",\n                \"source_dataset\": \"MLAAD-tiny\",\n            }\n        )\n        counts[label] += 1\n\n        if all(count >= max_files_per_class for count in counts.values()):\n            break\n\n    manifest = pd.DataFrame(records)\n    if set(manifest.get(\"label\", pd.Series(dtype=int)).unique()) != {0, 1}:\n        raise RuntimeError(\n            \"MLAAD-tiny smoke manifest could not collect both classes. \"\n            \"Provide GENUINE_AUDIO_DIR or inspect the MLAAD-tiny label/path fields.\"\n        )\n    manifest = manifest.sample(frac=1.0, random_state=random_state).reset_index(drop=True)\n    manifest.to_csv(manifest_path, index=False)\n    print(f\"Saved MLAAD-tiny smoke manifest: {manifest_path}\")\n    return manifest\n\n\ndef apply_smoke_limit(manifest: pd.DataFrame, max_files_per_class: int, random_state: int) -> pd.DataFrame:\n    if max_files_per_class is None or max_files_per_class <= 0:\n        return manifest.reset_index(drop=True)\n    frames = []\n    for label, group in manifest.groupby(\"label\", sort=True):\n        sample_n = min(max_files_per_class, len(group))\n        frames.append(group.sample(n=sample_n, random_state=random_state))\n    return pd.concat(frames, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)\n\n\ndef build_audio_manifest(\n    mlaad_synthetic_dir: str | Path,\n    genuine_audio_dir: Optional[str | Path],\n    smoke_test: bool,\n    max_files_per_class: int,\n    random_state: int,\n    require_genuine: bool,\n) -> pd.DataFrame:\n    \"\"\"Build the binary audio manifest from synthetic MLAAD and genuine audio.\"\"\"\n    synthetic = scan_synthetic_audio(mlaad_synthetic_dir)\n    frames = [synthetic]\n\n    if genuine_audio_dir:\n        frames.append(scan_genuine_audio(genuine_audio_dir))\n    elif require_genuine:\n        raise RuntimeError(\n            \"GENUINE_AUDIO_DIR is required for final binary training. \"\n            \"MLAAD v10 contains synthetic audio only; provide the matching M-AILABS/genuine Drive folder.\"\n        )\n\n    manifest = pd.concat(frames, ignore_index=True, sort=False)\n    manifest[\"label\"] = manifest[\"label\"].astype(int)\n    manifest[\"class_name\"] = manifest[\"label\"].map(CLASS_NAMES)\n    manifest[\"file_exists\"] = manifest[\"path\"].map(file_exists_bool)\n    manifest = manifest[manifest[\"file_exists\"]].copy()\n    if smoke_test:\n        manifest = apply_smoke_limit(manifest, max_files_per_class, random_state)\n    if set(manifest[\"label\"].unique()) != {0, 1}:\n        raise RuntimeError(\n            \"The manifest must contain both classes: 0=bona-fide/genuine and 1=synthetic/deepfake.\"\n        )\n    return manifest.reset_index(drop=True)\n\n\ndef build_dataset_summary(manifest: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for label, group in manifest.groupby(\"label\", sort=True):\n        rows.append(\n            {\n                \"class_label\": int(label),\n                \"class_name\": CLASS_NAMES[int(label)],\n                \"files\": int(len(group)),\n                \"languages\": int(group[\"language\"].nunique(dropna=True)),\n                \"tts_generators\": int(group[\"tts_generator\"].nunique(dropna=True)) if label == 1 else 0,\n                \"duration_hours_metadata\": float(pd.to_numeric(group.get(\"duration_metadata\"), errors=\"coerce\").sum() / 3600.0),\n            }\n        )\n    total = {\n        \"class_label\": \"all\",\n        \"class_name\": \"all\",\n        \"files\": int(len(manifest)),\n        \"languages\": int(manifest[\"language\"].nunique(dropna=True)),\n        \"tts_generators\": int(manifest.loc[manifest[\"label\"] == 1, \"tts_generator\"].nunique(dropna=True)),\n        \"duration_hours_metadata\": float(pd.to_numeric(manifest.get(\"duration_metadata\"), errors=\"coerce\").sum() / 3600.0),\n    }\n    rows.append(total)\n    return pd.DataFrame(rows)\n\n\ndef build_data_quality_checks(manifest: pd.DataFrame) -> pd.DataFrame:\n    checks = [\n        (\"total_rows\", len(manifest)),\n        (\"missing_paths\", manifest[\"path\"].isna().sum()),\n        (\"missing_files\", (~manifest[\"path\"].map(file_exists_bool)).sum()),\n        (\"missing_language\", (manifest[\"language\"].fillna(\"unknown\") == \"unknown\").sum()),\n        (\"missing_tts_generator_synthetic\", (manifest.loc[manifest[\"label\"] == 1, \"tts_generator\"].fillna(\"unknown\") == \"unknown\").sum()),\n        (\"missing_original_file_synthetic\", (manifest.loc[manifest[\"label\"] == 1, \"original_file\"].fillna(\"unknown\") == \"unknown\").sum()),\n        (\"duplicate_paths\", manifest[\"path\"].duplicated().sum()),\n    ]\n    return pd.DataFrame([{\"check\": name, \"value\": int(value)} for name, value in checks])\n\n\ndef plot_language_class_distribution(manifest: pd.DataFrame, output_path: str | Path, top_n: int = 20) -> None:\n    import matplotlib.pyplot as plt\n    import seaborn as sns\n\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    plot_df = manifest.copy()\n    top_languages = plot_df[\"language\"].value_counts().head(top_n).index\n    plot_df = plot_df[plot_df[\"language\"].isin(top_languages)]\n    counts = plot_df.groupby([\"language\", \"class_name\"]).size().reset_index(name=\"files\")\n    plt.figure(figsize=(12, 6))\n    sns.barplot(data=counts, x=\"language\", y=\"files\", hue=\"class_name\")\n    plt.title(\"Figure 1: Language and class distribution\")\n    plt.xlabel(\"Language\")\n    plt.ylabel(\"Files\")\n    plt.xticks(rotation=45, ha=\"right\")\n    plt.tight_layout()\n    plt.savefig(output_path, dpi=300, bbox_inches=\"tight\")\n    plt.show()\n\n\ndef _choose_generators(generators: Sequence[str], fraction: float, random_state: int) -> List[str]:\n    rng = random.Random(random_state)\n    generators = sorted([g for g in generators if _safe_str(g) != \"unknown\"])\n    if not generators:\n        return []\n    count = max(1, int(round(len(generators) * fraction)))\n    count = min(count, max(1, len(generators) - 1)) if len(generators) > 1 else 1\n    return sorted(rng.sample(generators, count))\n\n\ndef _assign_random_splits(df: pd.DataFrame, val_fraction: float, test_fraction: float, random_state: int) -> pd.Series:\n    rng = np.random.default_rng(random_state)\n    indices = np.array(df.index)\n    rng.shuffle(indices)\n    n = len(indices)\n    n_test = max(1, int(round(n * test_fraction))) if n >= 3 else max(0, n - 1)\n    n_val = max(1, int(round(n * val_fraction))) if n - n_test >= 3 else max(0, n - n_test - 1)\n    split = pd.Series(\"train\", index=df.index, dtype=\"object\")\n    split.loc[indices[:n_test]] = \"test\"\n    split.loc[indices[n_test:n_test + n_val]] = \"validation\"\n    return split\n\n\ndef make_tts_holdout_split(\n    manifest: pd.DataFrame,\n    val_fraction: float = 0.2,\n    test_fraction: float = 0.2,\n    random_state: int = 42,\n    enforce_original_file_exclusion: bool = True,\n) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    \"\"\"Create train/validation/test splits with unseen synthetic TTS generators in test.\"\"\"\n    df = manifest.reset_index(drop=True).copy()\n    df[\"split\"] = \"train\"\n    limitations = []\n\n    synthetic_mask = df[\"label\"] == 1\n    synthetic = df[synthetic_mask].copy()\n    generators = sorted(synthetic[\"tts_generator\"].fillna(\"unknown\").unique())\n\n    if len(generators) >= 3:\n        test_generators = _choose_generators(generators, test_fraction, random_state)\n        remaining_generators = [g for g in generators if g not in test_generators]\n        validation_generators = _choose_generators(remaining_generators, val_fraction, random_state + 1)\n\n        strict_original_split_applied = False\n        if enforce_original_file_exclusion and \"original_file\" in df.columns:\n            original_series = (\n                df.loc[synthetic_mask, \"original_file\"]\n                .dropna()\n                .astype(str)\n            )\n            originals = sorted(original_series[original_series != \"unknown\"].unique())\n            if len(originals) >= 3:\n                test_originals = set(_choose_generators(originals, test_fraction, random_state + 11))\n                remaining_originals = [o for o in originals if o not in test_originals]\n                validation_originals = set(_choose_generators(remaining_originals, val_fraction, random_state + 12))\n                heldout_originals = test_originals | validation_originals\n                original_values = df[\"original_file\"].astype(str)\n\n                test_mask = synthetic_mask & df[\"tts_generator\"].isin(test_generators) & original_values.isin(test_originals)\n                validation_mask = synthetic_mask & df[\"tts_generator\"].isin(validation_generators) & original_values.isin(validation_originals)\n                train_mask = (\n                    synthetic_mask\n                    & ~df[\"tts_generator\"].isin(test_generators + validation_generators)\n                    & ~original_values.isin(heldout_originals)\n                    & (original_values != \"unknown\")\n                )\n\n                if test_mask.any() and validation_mask.any() and train_mask.any():\n                    df.loc[synthetic_mask, \"split\"] = \"excluded_split_policy\"\n                    df.loc[train_mask, \"split\"] = \"train\"\n                    df.loc[validation_mask, \"split\"] = \"validation\"\n                    df.loc[test_mask, \"split\"] = \"test\"\n                    strict_original_split_applied = True\n                    excluded_count = int((df.loc[synthetic_mask, \"split\"] == \"excluded_split_policy\").sum())\n                    if excluded_count:\n                        limitations.append(\n                            f\"Excluded {excluded_count} synthetic rows to keep both held-out TTS generators and original_file groups separate.\"\n                        )\n                else:\n                    limitations.append(\n                        \"Could not jointly partition TTS generators and original_file groups without empty synthetic splits; using generator holdout and reporting original_file overlap.\"\n                    )\n\n        if not strict_original_split_applied:\n            df.loc[synthetic_mask & df[\"tts_generator\"].isin(test_generators), \"split\"] = \"test\"\n            df.loc[synthetic_mask & df[\"tts_generator\"].isin(validation_generators), \"split\"] = \"validation\"\n\n            if enforce_original_file_exclusion and \"original_file\" in df.columns:\n                test_original_series = (\n                    df.loc[(df[\"split\"] == \"test\") & synthetic_mask, \"original_file\"]\n                    .dropna()\n                    .astype(str)\n                )\n                test_originals = set(test_original_series[test_original_series != \"unknown\"])\n                overlap_mask = (\n                    synthetic_mask\n                    & df[\"split\"].isin([\"train\", \"validation\"])\n                    & df[\"original_file\"].astype(str).isin(test_originals)\n                )\n                if overlap_mask.any():\n                    candidate = df.copy()\n                    candidate.loc[overlap_mask, \"split\"] = \"excluded_original_overlap\"\n                    synthetic_counts = candidate[candidate[\"label\"] == 1].groupby(\"split\").size()\n                    required_splits_present = all(int(synthetic_counts.get(split_name, 0)) > 0 for split_name in (\"train\", \"validation\", \"test\"))\n                    if required_splits_present:\n                        df = candidate\n                        limitations.append(\n                            f\"Excluded {int(overlap_mask.sum())} synthetic rows because original_file also appeared in held-out test generators.\"\n                        )\n                    else:\n                        limitations.append(\n                            \"Original_file exclusion would remove too much synthetic data; kept generator holdout and reported original_file overlap.\"\n                        )\n    else:\n        df.loc[synthetic.index, \"split\"] = _assign_random_splits(synthetic, val_fraction, test_fraction, random_state)\n        limitations.append(\"Too few synthetic TTS generators for strict generator holdout; synthetic rows used random split.\")\n\n    genuine = df[df[\"label\"] == 0].copy()\n    if not genuine.empty:\n        df.loc[genuine.index, \"split\"] = _assign_random_splits(genuine, val_fraction, test_fraction, random_state + 2)\n\n    report_rows = []\n    for split_name, group in df.groupby(\"split\", sort=False):\n        report_rows.append(\n            {\n                \"split\": split_name,\n                \"rows\": int(len(group)),\n                \"bona_fide\": int((group[\"label\"] == 0).sum()),\n                \"synthetic\": int((group[\"label\"] == 1).sum()),\n                \"languages\": int(group[\"language\"].nunique(dropna=True)),\n                \"synthetic_tts_generators\": int(group.loc[group[\"label\"] == 1, \"tts_generator\"].nunique(dropna=True)),\n            }\n        )\n    for limitation in limitations:\n        report_rows.append({\"split\": \"limitation\", \"rows\": 0, \"bona_fide\": 0, \"synthetic\": 0, \"languages\": 0, \"synthetic_tts_generators\": 0, \"note\": limitation})\n    return df, pd.DataFrame(report_rows)\n\n\ndef compute_leakage_checks(split_manifest: pd.DataFrame) -> pd.DataFrame:\n    usable = split_manifest[split_manifest[\"split\"].isin([\"train\", \"validation\", \"test\"])].copy()\n    rows = []\n\n    train_val_generators = set(\n        usable.loc[(usable[\"label\"] == 1) & usable[\"split\"].isin([\"train\", \"validation\"]), \"tts_generator\"].dropna().astype(str)\n    )\n    test_generators = set(\n        usable.loc[(usable[\"label\"] == 1) & (usable[\"split\"] == \"test\"), \"tts_generator\"].dropna().astype(str)\n    )\n    generator_overlap = sorted(train_val_generators & test_generators)\n    rows.append(\n        {\n            \"check\": \"synthetic_tts_generator_overlap_trainval_test\",\n            \"value\": len(generator_overlap),\n            \"status\": \"pass\" if not generator_overlap else \"review\",\n            \"details\": \", \".join(generator_overlap[:20]),\n        }\n    )\n\n    for left, right in ((\"train\", \"validation\"), (\"train\", \"test\"), (\"validation\", \"test\")):\n        left_paths = set(usable.loc[usable[\"split\"] == left, \"path\"].astype(str))\n        right_paths = set(usable.loc[usable[\"split\"] == right, \"path\"].astype(str))\n        rows.append(\n            {\n                \"check\": f\"path_overlap_{left}_{right}\",\n                \"value\": len(left_paths & right_paths),\n                \"status\": \"pass\" if not (left_paths & right_paths) else \"fail\",\n                \"details\": \"\",\n            }\n        )\n\n    if \"original_file\" in usable.columns:\n        for left, right in ((\"train\", \"validation\"), (\"train\", \"test\"), (\"validation\", \"test\")):\n            left_keys = set(usable.loc[usable[\"split\"] == left, \"original_file\"].dropna().astype(str)) - {\"unknown\"}\n            right_keys = set(usable.loc[usable[\"split\"] == right, \"original_file\"].dropna().astype(str)) - {\"unknown\"}\n            overlap = sorted(left_keys & right_keys)\n            rows.append(\n                {\n                    \"check\": f\"original_file_overlap_{left}_{right}\",\n                    \"value\": len(overlap),\n                    \"status\": \"pass\" if not overlap else \"review\",\n                    \"details\": \", \".join(overlap[:20]),\n                }\n            )\n\n    for split_name, group in usable.groupby(\"split\", sort=True):\n        labels = set(group[\"label\"].astype(int).unique())\n        rows.append(\n            {\n                \"check\": f\"{split_name}_contains_both_classes\",\n                \"value\": int(labels == {0, 1}),\n                \"status\": \"pass\" if labels == {0, 1} else \"fail\",\n                \"details\": f\"labels={sorted(labels)}\",\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef build_feature_config(\n    sample_rate: int = 22050,\n    fixed_duration_seconds: float = 5.0,\n    n_mfcc: int = 40,\n    n_fft: int = 1024,\n    hop_length: Optional[int] = None,\n    win_length: Optional[int] = None,\n) -> dict:\n    if hop_length is None:\n        hop_length = int(round(sample_rate * 0.010))\n    if win_length is None:\n        win_length = int(round(sample_rate * 0.025))\n    max_frames = max(1, 1 + max(0, int(sample_rate * fixed_duration_seconds) - n_fft) // hop_length)\n    return {\n        \"sample_rate\": int(sample_rate),\n        \"fixed_duration_seconds\": float(fixed_duration_seconds),\n        \"n_mfcc\": int(n_mfcc),\n        \"n_fft\": int(n_fft),\n        \"hop_length\": int(hop_length),\n        \"win_length\": int(win_length),\n        \"max_frames\": int(max_frames),\n        \"aggregation\": \"mean_std\",\n    }\n\n\ndef _load_audio_fixed(path: str | Path, config: dict) -> np.ndarray:\n    import librosa\n\n    sample_rate = int(config[\"sample_rate\"])\n    target_samples = int(round(sample_rate * float(config[\"fixed_duration_seconds\"])))\n    audio, _ = librosa.load(str(path), sr=sample_rate, mono=True)\n    audio = np.asarray(audio, dtype=np.float32)\n    if len(audio) < target_samples:\n        audio = np.pad(audio, (0, target_samples - len(audio)))\n    elif len(audio) > target_samples:\n        audio = audio[:target_samples]\n    return audio\n\n\ndef extract_mfcc_pair(path: str | Path, config: dict) -> Tuple[np.ndarray, np.ndarray]:\n    import librosa\n\n    audio = _load_audio_fixed(path, config)\n    mfcc = librosa.feature.mfcc(\n        y=audio,\n        sr=int(config[\"sample_rate\"]),\n        n_mfcc=int(config[\"n_mfcc\"]),\n        n_fft=int(config[\"n_fft\"]),\n        hop_length=int(config[\"hop_length\"]),\n        win_length=int(config[\"win_length\"]),\n        center=False,\n    ).T.astype(np.float32)\n\n    max_frames = int(config[\"max_frames\"])\n    if mfcc.shape[0] < max_frames:\n        pad = np.zeros((max_frames - mfcc.shape[0], mfcc.shape[1]), dtype=np.float32)\n        mfcc = np.vstack([mfcc, pad])\n    elif mfcc.shape[0] > max_frames:\n        mfcc = mfcc[:max_frames]\n\n    agg = np.concatenate([mfcc.mean(axis=0), mfcc.std(axis=0)]).astype(np.float32)\n    return agg, mfcc\n\n\ndef _append_h5_batch(h5, agg_batch, seq_batch, y_batch, manifest_rows):\n    n_old = h5[\"X_agg\"].shape[0]\n    n_new = n_old + len(y_batch)\n    for name in (\"X_agg\", \"X_seq\", \"y\", \"manifest_row\"):\n        h5[name].resize((n_new, *h5[name].shape[1:]))\n    h5[\"X_agg\"][n_old:n_new] = np.asarray(agg_batch, dtype=np.float32)\n    h5[\"X_seq\"][n_old:n_new] = np.asarray(seq_batch, dtype=np.float32)\n    h5[\"y\"][n_old:n_new] = np.asarray(y_batch, dtype=np.int8)\n    h5[\"manifest_row\"][n_old:n_new] = np.asarray(manifest_rows, dtype=np.int64)\n\n\ndef build_feature_cache(\n    split_manifest: pd.DataFrame,\n    feature_h5_path: str | Path,\n    feature_index_path: str | Path,\n    feature_config_path: str | Path,\n    config: dict,\n    force_rebuild: bool = False,\n    batch_write_size: int = 64,\n) -> Tuple[Path, Path]:\n    \"\"\"Extract MFCCs once and cache aggregate plus sequence features to HDF5.\"\"\"\n    import h5py\n    from tqdm.auto import tqdm\n\n    feature_h5_path = Path(feature_h5_path)\n    feature_index_path = Path(feature_index_path)\n    feature_config_path = Path(feature_config_path)\n    expected_hash = stable_config_hash(config)\n\n    if feature_h5_path.exists() and feature_index_path.exists() and feature_config_path.exists() and not force_rebuild:\n        existing = read_json(feature_config_path)\n        if existing.get(\"config_hash\") == expected_hash:\n            print(f\"Reusing MFCC feature cache: {feature_h5_path}\")\n            return feature_h5_path, feature_index_path\n\n    feature_h5_path.parent.mkdir(parents=True, exist_ok=True)\n    feature_index_path.parent.mkdir(parents=True, exist_ok=True)\n    usable = split_manifest[split_manifest[\"split\"].isin([\"train\", \"validation\", \"test\"])].copy().reset_index(drop=False)\n    if usable.empty:\n        raise RuntimeError(\"No train/validation/test rows are available for feature extraction.\")\n\n    if feature_h5_path.exists():\n        feature_h5_path.unlink()\n    if feature_index_path.exists():\n        feature_index_path.unlink()\n\n    n_mfcc = int(config[\"n_mfcc\"])\n    max_frames = int(config[\"max_frames\"])\n    index_rows = []\n    errors = []\n\n    with h5py.File(feature_h5_path, \"w\") as h5:\n        h5.attrs[\"config_hash\"] = expected_hash\n        h5.create_dataset(\"X_agg\", shape=(0, 2 * n_mfcc), maxshape=(None, 2 * n_mfcc), dtype=\"float32\", chunks=True)\n        h5.create_dataset(\"X_seq\", shape=(0, max_frames, n_mfcc), maxshape=(None, max_frames, n_mfcc), dtype=\"float32\", chunks=True)\n        h5.create_dataset(\"y\", shape=(0,), maxshape=(None,), dtype=\"int8\", chunks=True)\n        h5.create_dataset(\"manifest_row\", shape=(0,), maxshape=(None,), dtype=\"int64\", chunks=True)\n\n        agg_batch, seq_batch, y_batch, manifest_rows = [], [], [], []\n        feature_row = 0\n        for _, row in tqdm(usable.iterrows(), total=len(usable), desc=\"Extracting MFCC features\"):\n            try:\n                agg, seq = extract_mfcc_pair(row[\"path\"], config)\n            except Exception as exc:\n                errors.append({\"path\": row[\"path\"], \"error\": repr(exc)})\n                continue\n\n            agg_batch.append(agg)\n            seq_batch.append(seq)\n            y_batch.append(int(row[\"label\"]))\n            manifest_rows.append(int(row[\"index\"]))\n            row_dict = row.drop(labels=[\"index\"]).to_dict()\n            row_dict[\"feature_row\"] = feature_row\n            index_rows.append(row_dict)\n            feature_row += 1\n\n            if len(y_batch) >= batch_write_size:\n                _append_h5_batch(h5, agg_batch, seq_batch, y_batch, manifest_rows)\n                agg_batch, seq_batch, y_batch, manifest_rows = [], [], [], []\n\n        if y_batch:\n            _append_h5_batch(h5, agg_batch, seq_batch, y_batch, manifest_rows)\n\n    feature_index = pd.DataFrame(index_rows)\n    feature_index.to_csv(feature_index_path, index=False)\n    write_json(feature_config_path, {\"config_hash\": expected_hash, \"feature_config\": config})\n    if errors:\n        pd.DataFrame(errors).to_csv(feature_index_path.with_name(\"feature_extraction_errors.csv\"), index=False)\n    print(f\"Saved MFCC feature cache: {feature_h5_path}\")\n    print(f\"Saved feature index: {feature_index_path}\")\n    return feature_h5_path, feature_index_path\n\n\ndef build_preprocessing_summary(config: dict, feature_index: pd.DataFrame) -> pd.DataFrame:\n    rows = [\n        (\"sample_rate\", config[\"sample_rate\"]),\n        (\"fixed_duration_seconds\", config[\"fixed_duration_seconds\"]),\n        (\"n_mfcc\", config[\"n_mfcc\"]),\n        (\"n_fft\", config[\"n_fft\"]),\n        (\"hop_length\", config[\"hop_length\"]),\n        (\"win_length\", config[\"win_length\"]),\n        (\"max_frames\", config[\"max_frames\"]),\n        (\"aggregated_feature_count\", 2 * int(config[\"n_mfcc\"])),\n        (\"cached_files\", len(feature_index)),\n    ]\n    return pd.DataFrame(rows, columns=[\"setting\", \"value\"])\n\n\ndef read_hdf5_rows(h5_path: str | Path, dataset_name: str, rows: Sequence[int]) -> np.ndarray:\n    import h5py\n\n    rows = np.asarray(rows, dtype=np.int64)\n    if rows.size == 0:\n        raise ValueError(f\"No rows requested for dataset {dataset_name}.\")\n    order = np.argsort(rows)\n    sorted_rows = rows[order]\n    with h5py.File(h5_path, \"r\") as h5:\n        data_sorted = h5[dataset_name][sorted_rows]\n    inverse = np.argsort(order)\n    return data_sorted[inverse]\n\n\ndef load_feature_index(feature_index_path: str | Path) -> pd.DataFrame:\n    index = pd.read_csv(feature_index_path)\n    index[\"label\"] = index[\"label\"].astype(int)\n    index[\"feature_row\"] = index[\"feature_row\"].astype(int)\n    return index\n\n\ndef load_aggregated_split(feature_h5_path: str | Path, feature_index: pd.DataFrame, split: str) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:\n    subset = feature_index[feature_index[\"split\"] == split].copy().reset_index(drop=True)\n    X = read_hdf5_rows(feature_h5_path, \"X_agg\", subset[\"feature_row\"].to_numpy())\n    y = subset[\"label\"].to_numpy(dtype=np.int64)\n    return X, y, subset\n\n\ndef load_sequence_split(feature_h5_path: str | Path, feature_index: pd.DataFrame, split: str) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:\n    subset = feature_index[feature_index[\"split\"] == split].copy().reset_index(drop=True)\n    X = read_hdf5_rows(feature_h5_path, \"X_seq\", subset[\"feature_row\"].to_numpy())\n    y = subset[\"label\"].to_numpy(dtype=np.int64)\n    return X, y, subset\n\n\ndef undersample_binary_arrays(X: np.ndarray, y: np.ndarray, random_state: int) -> Tuple[np.ndarray, np.ndarray]:\n    rng = np.random.default_rng(random_state)\n    classes, counts = np.unique(y, return_counts=True)\n    if len(classes) != 2:\n        return X, y\n    n = int(counts.min())\n    selected = []\n    for cls in classes:\n        cls_idx = np.where(y == cls)[0]\n        selected.extend(rng.choice(cls_idx, size=n, replace=False).tolist())\n    selected = np.asarray(selected, dtype=np.int64)\n    rng.shuffle(selected)\n    return X[selected], y[selected]\n\n\ndef make_class_weight_dict(y: np.ndarray) -> Dict[int, float]:\n    from sklearn.utils.class_weight import compute_class_weight\n\n    classes = np.unique(y)\n    weights = compute_class_weight(class_weight=\"balanced\", classes=classes, y=y)\n    return {int(cls): float(weight) for cls, weight in zip(classes, weights)}\n\n\ndef fit_sequence_standard_scaler(\n    feature_h5_path: str | Path,\n    feature_index: pd.DataFrame,\n    split: str = \"train\",\n    max_files_for_scaler: int = 5000,\n    random_state: int = 42,\n):\n    from sklearn.preprocessing import StandardScaler\n\n    subset = feature_index[feature_index[\"split\"] == split].copy()\n    if len(subset) > max_files_for_scaler:\n        subset = subset.sample(n=max_files_for_scaler, random_state=random_state)\n    X = read_hdf5_rows(feature_h5_path, \"X_seq\", subset[\"feature_row\"].to_numpy())\n    scaler = StandardScaler()\n    scaler.fit(X.reshape(-1, X.shape[-1]))\n    return scaler\n\n\ndef sequence_batch_generator(\n    feature_h5_path: str | Path,\n    feature_rows: Sequence[int],\n    labels: Sequence[int],\n    batch_size: int,\n    scaler=None,\n    shuffle: bool = False,\n    random_state: int = 42,\n) -> Iterable[Tuple[np.ndarray, np.ndarray]]:\n    rows = np.asarray(feature_rows, dtype=np.int64)\n    labels = np.asarray(labels, dtype=np.int64)\n    rng = np.random.default_rng(random_state)\n    order = np.arange(len(rows))\n    if shuffle:\n        rng.shuffle(order)\n    rows = rows[order]\n    labels = labels[order]\n    for start in range(0, len(rows), batch_size):\n        end = start + batch_size\n        batch_rows = rows[start:end]\n        batch_labels = labels[start:end]\n        X_batch = read_hdf5_rows(feature_h5_path, \"X_seq\", batch_rows).astype(np.float32)\n        if scaler is not None:\n            original_shape = X_batch.shape\n            X_batch = scaler.transform(X_batch.reshape(-1, original_shape[-1])).reshape(original_shape).astype(np.float32)\n        yield X_batch, batch_labels\n\n\ndef metrics_from_predictions(\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    y_score: Optional[np.ndarray],\n    model_name: str,\n    variant: str,\n    split: str,\n) -> dict:\n    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score\n\n    row = {\n        \"model\": model_name,\n        \"variant\": variant,\n        \"split\": split,\n        \"accuracy\": float(accuracy_score(y_true, y_pred)),\n        \"precision\": float(precision_score(y_true, y_pred, zero_division=0)),\n        \"recall\": float(recall_score(y_true, y_pred, zero_division=0)),\n        \"f1\": float(f1_score(y_true, y_pred, zero_division=0)),\n    }\n    if y_score is not None and len(np.unique(y_true)) == 2:\n        try:\n            row[\"roc_auc\"] = float(roc_auc_score(y_true, y_score))\n        except Exception:\n            row[\"roc_auc\"] = np.nan\n    else:\n        row[\"roc_auc\"] = np.nan\n    return row\n\n\ndef classifier_scores(model, X: np.ndarray) -> Tuple[np.ndarray, Optional[np.ndarray]]:\n    if hasattr(model, \"predict_proba\"):\n        probs = model.predict_proba(X)\n        if probs.ndim == 2 and probs.shape[1] >= 2:\n            return np.argmax(probs, axis=1), probs[:, 1]\n    y_pred = model.predict(X)\n    if hasattr(model, \"decision_function\"):\n        score = model.decision_function(X)\n        if np.asarray(score).ndim > 1:\n            score = np.asarray(score)[:, -1]\n        return y_pred, np.asarray(score)\n    return y_pred, None\n\n\ndef upsert_rows(csv_path: str | Path, rows: List[dict], key_columns: Sequence[str]) -> pd.DataFrame:\n    csv_path = Path(csv_path)\n    csv_path.parent.mkdir(parents=True, exist_ok=True)\n    new_df = pd.DataFrame(rows)\n    if csv_path.exists():\n        existing = pd.read_csv(csv_path)\n        for _, row in new_df.iterrows():\n            mask = pd.Series(True, index=existing.index)\n            for key in key_columns:\n                mask &= existing[key].astype(str) == str(row[key])\n            existing = existing.loc[~mask]\n        out = pd.concat([existing, new_df], ignore_index=True, sort=False)\n    else:\n        out = new_df\n    out.to_csv(csv_path, index=False)\n    return out\n\n\ndef save_classification_report(path: str | Path, y_true: np.ndarray, y_pred: np.ndarray, model_name: str, split: str) -> pd.DataFrame:\n    from sklearn.metrics import classification_report\n\n    report = classification_report(\n        y_true,\n        y_pred,\n        target_names=[CLASS_NAMES[0], CLASS_NAMES[1]],\n        output_dict=True,\n        zero_division=0,\n    )\n    rows = []\n    for label, metrics in report.items():\n        if isinstance(metrics, dict):\n            row = {\"model\": model_name, \"split\": split, \"label\": label}\n            row.update(metrics)\n            rows.append(row)\n    df = pd.DataFrame(rows)\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(path, index=False)\n    return df\n\n\ndef plot_confusion_matrix(\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    output_path: str | Path,\n    title: str,\n) -> None:\n    import matplotlib.pyplot as plt\n    import seaborn as sns\n    from sklearn.metrics import confusion_matrix\n\n    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    plt.figure(figsize=(5.5, 4.5))\n    sns.heatmap(\n        cm,\n        annot=True,\n        fmt=\"d\",\n        cmap=\"Blues\",\n        xticklabels=[CLASS_NAMES[0], CLASS_NAMES[1]],\n        yticklabels=[CLASS_NAMES[0], CLASS_NAMES[1]],\n    )\n    plt.title(title)\n    plt.xlabel(\"Predicted label\")\n    plt.ylabel(\"True label\")\n    plt.tight_layout()\n    plt.savefig(output_path, dpi=300, bbox_inches=\"tight\")\n    plt.show()\n\n\ndef plot_training_curves(history, output_path: str | Path, title: str) -> None:\n    import matplotlib.pyplot as plt\n\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    hist = history.history if hasattr(history, \"history\") else history\n    plt.figure(figsize=(10, 4))\n    plt.subplot(1, 2, 1)\n    plt.plot(hist.get(\"loss\", []), label=\"train_loss\")\n    plt.plot(hist.get(\"val_loss\", []), label=\"val_loss\")\n    plt.title(f\"{title}: loss\")\n    plt.xlabel(\"Epoch\")\n    plt.legend()\n    plt.subplot(1, 2, 2)\n    plt.plot(hist.get(\"accuracy\", []), label=\"train_accuracy\")\n    plt.plot(hist.get(\"val_accuracy\", []), label=\"val_accuracy\")\n    plt.title(f\"{title}: accuracy\")\n    plt.xlabel(\"Epoch\")\n    plt.legend()\n    plt.tight_layout()\n    plt.savefig(output_path, dpi=300, bbox_inches=\"tight\")\n    plt.show()\n\n\ndef evaluate_by_group(\n    metadata: pd.DataFrame,\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    group_col: str,\n    min_rows: int = 5,\n) -> pd.DataFrame:\n    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score\n\n    df = metadata.copy().reset_index(drop=True)\n    df[\"y_true\"] = y_true\n    df[\"y_pred\"] = y_pred\n    rows = []\n    for group, group_df in df.groupby(group_col):\n        if len(group_df) < min_rows:\n            continue\n        rows.append(\n            {\n                group_col: group,\n                \"rows\": int(len(group_df)),\n                \"accuracy\": float(accuracy_score(group_df[\"y_true\"], group_df[\"y_pred\"])),\n                \"precision\": float(precision_score(group_df[\"y_true\"], group_df[\"y_pred\"], zero_division=0)),\n                \"recall\": float(recall_score(group_df[\"y_true\"], group_df[\"y_pred\"], zero_division=0)),\n                \"f1\": float(f1_score(group_df[\"y_true\"], group_df[\"y_pred\"], zero_division=0)),\n            }\n        )\n    return pd.DataFrame(rows).sort_values([\"f1\", \"rows\"], ascending=[True, False])\n\n\ndef plot_group_metric(group_df: pd.DataFrame, group_col: str, output_path: str | Path, title: str, top_n: int = 15) -> None:\n    import matplotlib.pyplot as plt\n    import seaborn as sns\n\n    if group_df.empty:\n        print(f\"No group-level rows available for {group_col}.\")\n        return\n    plot_df = group_df.head(top_n).copy()\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    plt.figure(figsize=(10, 5))\n    sns.barplot(data=plot_df, y=group_col, x=\"f1\", color=\"#4c78a8\")\n    plt.title(title)\n    plt.xlabel(\"F1 score\")\n    plt.ylabel(group_col)\n    plt.xlim(0, 1)\n    plt.tight_layout()\n    plt.savefig(output_path, dpi=300, bbox_inches=\"tight\")\n    plt.show()\n\n\ndef copy_helper_to_drive(helper_source_path: str | Path, output_root: str | Path) -> Path:\n    output_root = Path(output_root)\n    target = output_root / \"cached\" / \"mlaad_colab_common.py\"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(helper_source_path, target)\n    return target\n"
HELPER_PATH = Path("/content/mlaad_colab_common.py")
HELPER_PATH.write_text(HELPER_SOURCE, encoding="utf-8")
if str(HELPER_PATH.parent) not in sys.path:
    sys.path.insert(0, str(HELPER_PATH.parent))

from mlaad_colab_common import *

REQUIRED_PACKAGES = [('librosa', 'librosa'), ('soundfile', 'soundfile'), ('tqdm', 'tqdm'), ('seaborn', 'seaborn'), ('sklearn', 'scikit-learn'), ('h5py', 'h5py'), ('joblib', 'joblib'), ('datasets', 'datasets')]
if REQUIRED_PACKAGES:
    install_missing_packages(REQUIRED_PACKAGES)


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

mount_drive_if_colab()

SMOKE_TEST = True
MAX_FILES_PER_CLASS = 40
USE_MLAAD_TINY_FOR_SMOKE = True
FORCE_REBUILD_FEATURES = False

MLAAD_SYNTHETIC_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Education/Music/datasets/MLAAD_10pct")
GENUINE_AUDIO_DIR = None  # Set this to the M-AILABS/genuine Drive folder for final full training.
OUTPUT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Music/outputs/mlaad_deepfake_detection")

SAMPLE_RATE = 22050
FIXED_DURATION_SECONDS = 5
N_MFCC = 40
RANDOM_STATE = 42

dirs = ensure_output_dirs(OUTPUT_ROOT)
print("Output root:", OUTPUT_ROOT)
for name in ("figures", "cached", "mfcc", "metrics", "models", "tables"):
    print(f"{name}: {dirs[name]}")


## Data source checks


In [ ]:
if SMOKE_TEST and USE_MLAAD_TINY_FOR_SMOKE and GENUINE_AUDIO_DIR is None:
    print("SMOKE_TEST is enabled and no genuine Drive path is set.")
    print("Using MLAAD-tiny from Hugging Face for smoke testing only.")
    manifest = build_mlaad_tiny_smoke_manifest(
        output_root=OUTPUT_ROOT,
        max_files_per_class=MAX_FILES_PER_CLASS,
        random_state=RANDOM_STATE,
    )
else:
    if not SMOKE_TEST and GENUINE_AUDIO_DIR is None:
        raise RuntimeError(
            "Final full training requires GENUINE_AUDIO_DIR. "
            "MLAAD v10 contains synthetic audio only; provide the matching M-AILABS/genuine folder."
        )
    manifest = build_audio_manifest(
        mlaad_synthetic_dir=MLAAD_SYNTHETIC_DIR,
        genuine_audio_dir=GENUINE_AUDIO_DIR,
        smoke_test=SMOKE_TEST,
        max_files_per_class=MAX_FILES_PER_CLASS,
        random_state=RANDOM_STATE,
        require_genuine=not SMOKE_TEST,
    )

manifest_path = dirs["cached"] / "manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Saved manifest: {manifest_path}")
display(manifest.head())
display(manifest["class_name"].value_counts())


## Table 1 and Figure 1


In [ ]:
table1 = build_dataset_summary(manifest)
table1_path = dirs["tables"] / "table1_dataset_summary.csv"
table1.to_csv(table1_path, index=False)
print(f"Saved Table 1: {table1_path}")
display(table1)

quality = build_data_quality_checks(manifest)
quality_path = dirs["tables"] / "data_quality_checks.csv"
quality.to_csv(quality_path, index=False)
print(f"Saved data-quality checks: {quality_path}")
display(quality)

figure1_path = dirs["figures"] / "figure1_language_class_distribution.png"
plot_language_class_distribution(manifest, figure1_path)
print(f"Saved Figure 1: {figure1_path}")


## Leakage-aware split


In [ ]:
split_manifest, split_report = make_tts_holdout_split(
    manifest,
    val_fraction=0.2,
    test_fraction=0.2,
    random_state=RANDOM_STATE,
    enforce_original_file_exclusion=True,
)

split_manifest_path = dirs["cached"] / "manifest_with_splits.csv"
split_manifest.to_csv(split_manifest_path, index=False)
split_counts_path = dirs["tables"] / "train_validation_test_split_counts.csv"
split_report.to_csv(split_counts_path, index=False)

leakage_checks = compute_leakage_checks(split_manifest)
leakage_path = dirs["tables"] / "leakage_checks.csv"
leakage_checks.to_csv(leakage_path, index=False)

print(f"Saved split manifest: {split_manifest_path}")
print(f"Saved split counts: {split_counts_path}")
print(f"Saved leakage checks: {leakage_path}")
display(split_report)
display(leakage_checks)


## MFCC preprocessing cache


In [ ]:
feature_config = build_feature_config(
    sample_rate=SAMPLE_RATE,
    fixed_duration_seconds=FIXED_DURATION_SECONDS,
    n_mfcc=N_MFCC,
)
feature_h5_path = dirs["mfcc"] / "mfcc_features_sr22050_dur5_nmfcc40.h5"
feature_index_path = dirs["cached"] / "mfcc_feature_index.csv"
feature_config_path = dirs["cached"] / "mfcc_feature_config.json"

build_feature_cache(
    split_manifest=split_manifest,
    feature_h5_path=feature_h5_path,
    feature_index_path=feature_index_path,
    feature_config_path=feature_config_path,
    config=feature_config,
    force_rebuild=FORCE_REBUILD_FEATURES,
)

feature_index = load_feature_index(feature_index_path)
preprocessing_summary = build_preprocessing_summary(feature_config, feature_index)
preprocessing_path = dirs["tables"] / "preprocessing_summary.csv"
preprocessing_summary.to_csv(preprocessing_path, index=False)
print(f"Saved preprocessing summary: {preprocessing_path}")
display(preprocessing_summary)
display(feature_index.groupby(["split", "class_name"]).size().unstack(fill_value=0))


## Checks

The next notebooks reuse the cached MFCC features:

- `01_svm_mfcc_baseline.ipynb`
- `02_mlp_mfcc_model.ipynb`
- `03_lstm_mfcc_sequence_model.ipynb`

Keep `SMOKE_TEST = True` until the smoke run completes. For final results, set `SMOKE_TEST = False` and provide a valid `GENUINE_AUDIO_DIR`.
